In [57]:
spark

StatementMeta(clusterSpark, 7, 40, Finished, Available, Finished)

In [58]:
from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.Path")
java_import(spark._jvm, "org.apache.hadoop.fs.FileSystem")
java_import(spark._jvm, "java.net.URI")

uri = spark._jvm.java.net.URI("abfss://bronze@dtalakegen2.dfs.core.windows.net/")
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    uri,
    spark._jsc.hadoopConfiguration()
)

files = fs.listStatus(
    spark._jvm.org.apache.hadoop.fs.Path(uri)
)

for file in files:
    print(file.getPath().toString())


StatementMeta(clusterSpark, 7, 41, Finished, Available, Finished)

abfss://bronze@dtalakegen2.dfs.core.windows.net/2000Q1.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2000Q2.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2000Q3.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2000Q4.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2001Q1.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2001Q2.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2001Q3.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2001Q4.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2002Q1.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2002Q2.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2002Q3.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2002Q4.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2003Q1.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2003Q2.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2003Q3.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2003Q4.csv
abfss://bronze@dtalakegen2.dfs.core.windows.net/2004Q1.c

In [21]:
sample_df = spark.read.csv(
    "abfss://bronze@dtalakegen2.dfs.core.windows.net/2024Q4.csv",
    header=False,
    inferSchema=True,
    sep="|"
)

schema = sample_df.schema

paths = [file.getPath().toString() for file in files]

for path in paths:
    
    if ".csv" in path.lower():
        
        print(f"Procesando {path}")
        
        df = spark.read.csv(
            path,
            header=False,
            schema=schema,
            sep="|"
        )
        
        df.write \
          .format("delta") \
          .mode("append") \
          .save("abfss://silver@dtalakegen2.dfs.core.windows.net/dataset/")

StatementMeta(clusterSpark, 7, 4, Finished, Cancelled, Cancelled)

Procesando abfss://bronze@dtalakegen2.dfs.core.windows.net/2000Q1.csv
Procesando abfss://bronze@dtalakegen2.dfs.core.windows.net/2000Q2.csv


---

In [59]:
silver_raw = spark.read.format("delta") \
    .load("abfss://silver@dtalakegen2.dfs.core.windows.net/dataset/")


StatementMeta(clusterSpark, 7, 42, Finished, Available, Finished)

In [60]:
silver_raw.count()

StatementMeta(clusterSpark, 7, 43, Finished, Available, Finished)

3182976596

In [74]:
silver_raw.show(5, truncate=False)

StatementMeta(clusterSpark, 7, 57, Finished, Available, Finished)

+----+----+------+---+---------------------+---------------------+----+---+---+--------+----+----+----+-----+-----+----+----+----+-----+----+----+----+----+----+----+----+----+----+----+----+----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|_c0 |_c1 |_c2   |_c3|_c4                  |_c5                  |_c6 |_c7|_c8|_c9     |_c10|_c11|_c12|_c13 |_c14 |_c15|_c16|_c17|_c18 |_c19|_c20|_c21|_c22|_c23|_c24|_c25|_c26|_c27|_c28|_c29|_c30|_c31 |_c32|_c33|_c34|_c35|_c36|_c37|_c38|_c39|_c40|_c41|_c42|_c43|_c44|_c45|_c46|_c47|_c48|_c49|_c50|_c51|_c52|_c53|_c54|_c55|_c56|_c57|_c58|_c59|_c60|_c61|_c62|_c63|_c64|_c65|_c66|_c67|_c68|_c69|_c70|

In [45]:
silver_raw.printSchema()

StatementMeta(clusterSpark, 7, 28, Finished, Available, Finished)

root
 |-- _c0: string (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: double (nullable = true)
 |-- _c8: double (nullable = true)
 |-- _c9: double (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: integer (nullable = true)
 |-- _c13: integer (nullable = true)
 |-- _c14: integer (nullable = true)
 |-- _c15: integer (nullable = true)
 |-- _c16: integer (nullable = true)
 |-- _c17: integer (nullable = true)
 |-- _c18: integer (nullable = true)
 |-- _c19: integer (nullable = true)
 |-- _c20: integer (nullable = true)
 |-- _c21: integer (nullable = true)
 |-- _c22: integer (nullable = true)
 |-- _c23: integer (nullable = true)
 |-- _c24: integer (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (nullable = true)
 |-- _c27: string 

In [75]:
from pyspark.sql.functions import col

dataset_modeling = silver_raw.select(
    # _c0 = Reference Pool ID (NA para SF → null, no lo necesitas)
    col("_c1").alias("loan_id"),              # Field Pos 2: Loan Identifier
    col("_c2").alias("reporting_period"),      # Field Pos 3: Monthly Reporting Period
    col("_c8").alias("current_interest_rate"), # Field Pos 9
    col("_c9").alias("orig_upb"),             # Field Pos 10: Original UPB
    col("_c11").alias("current_actual_upb"),   # Field Pos 12
    col("_c12").alias("original_term"),        # Field Pos 13
    col("_c13").alias("origination_date"),     # Field Pos 14
    col("_c15").alias("loan_age"),            # Field Pos 16
    col("_c16").alias("remaining_months_legal_maturity"),  # Field Pos 17
    col("_c17").alias("remaining_months_to_maturity"),     # Field Pos 18
    col("_c19").alias("orig_ltv"),            # Field Pos 20
    col("_c23").alias("credit_score"),        # Field Pos 24: Borrower Credit Score
    col("_c27").alias("property_type"),       # Field Pos 28
    col("_c30").alias("state"),               # Field Pos 31
    col("_c34").alias("amortization_type"),   # Field Pos 35
    col("_c39").alias("current_delinquency_status"),  # Field Pos 40
    col("_c40").alias("loan_payment_history"),        # Field Pos 41
    col("_c41").alias("modification_flag"),            # Field Pos 42
    col("_c43").alias("zero_balance_code"),            # Field Pos 44
    col("_c44").alias("zero_balance_effective_date"),  # Field Pos 45
    col("_c45").alias("upb_at_removal"),               # Field Pos 46
    col("_c50").alias("last_paid_installment_date"),   # Field Pos 51
    col("_c51").alias("foreclosure_date"),             # Field Pos 52
    col("_c52").alias("disposition_date"),             # Field Pos 53
    col("_c73").alias("servicing_activity_indicator"), # Field Pos 74
    col("_c101").alias("borrower_assistance_plan"),    # Field Pos 102
    col("_c105").alias("alt_delinquency_resolution"),  # Field Pos 106
    col("_c106").alias("alt_delinquency_resolution_count")  # Field Pos 107
)

StatementMeta(clusterSpark, 7, 58, Finished, Available, Finished)

In [50]:
# NO FUNCIONA
# orig_clean = orig_clean.filter(
#     (col("credit_score").between(300, 850)) &
#     (col("orig_ltv").between(10, 150)) &
#     (col("interest_rate") > 0) &
#     (col("orig_upb") > 10000)
# )

StatementMeta(clusterSpark, 7, 33, Finished, Available, Finished)

In [76]:
dataset_modeling = dataset_modeling.sample(fraction=0.05, seed=42)
dataset_modeling.count()

StatementMeta(clusterSpark, 7, 59, Finished, Available, Finished)

159149763

In [77]:
dataset_modeling.limit(200000).write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver@dtalakegen2.dfs.core.windows.net/orig_modeling/")


StatementMeta(clusterSpark, 7, 60, Finished, Available, Finished)

In [17]:
orig_sample.limit(200000).toPandas().to_csv("/tmp/orig_modeling_sample.csv", index=False)


StatementMeta(clusterSpark, 6, 12, Finished, Available, Finished)

In [18]:
orig_sample.limit(200000).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("abfss://silver@dtalakegen2.dfs.core.windows.net/export/orig_sample_csv/")


StatementMeta(clusterSpark, 6, 13, Finished, Available, Finished)